In [9]:
import xarray as xr
import os
import matplotlib.pyplot as plt
import numpy as np

datapath = '/Users/jukesliu/Documents/POSTDOC/snow-radar/data_submission/sample_data_RME/'

The NetCDF files are not self-describing - the variable names are not clear,  the variables contain no attributes other than the name of the data field, and there are no global attributes, including no grid mapping variable to describe the coordinate system/projection of the data. Moreover, the netcdf files do not plot correctly because the x variable contains NaNs (as do the z variables). If x, y, and z variables are to be coordinate variables, they cannot have NaNs.

In [129]:
for file in os.listdir(datapath):
    if file.endswith('.nc'):
        print(datapath+file)
        
        # open old dataset
        ds = xr.open_dataset(datapath+file, engine="netcdf4")
        
        # determine which coordinates to remove
        remove_x_idxs = np.where((np.isnan(ds.x) | np.isnan(ds.y)))[0]
        
        # grab the adjusted PDATA (remove parts of file with empty coordinates)
        PDATA = xr.DataArray(np.delete(ds.PDATA,remove_x_idxs,axis=1), 
                     coords={
                         'TWT': np.array(ds.TWT), # two way travel times
                         'x': np.arange(0, len(np.delete(ds.x, remove_x_idxs))) # x indices
                     },
                     dims=['TWT','x'],
                    )
        
        # grab the UTM xs and ys
        UTMx = xr.DataArray(np.delete(ds.x, remove_x_idxs),
                     coords={'x': np.arange(0, len(np.delete(ds.x, remove_x_idxs)))},
                     dims=['x'],)
        UTMy = xr.DataArray(np.delete(ds.y, remove_x_idxs),
                     coords={'x': np.arange(0, len(np.delete(ds.x, remove_x_idxs)))},
                     dims=['x'],)
        
        dataset = xr.Dataset({'PDATA': PDATA, 'UTMx': UTMx, 'UTMy': UTMy}) # combine all variables
        dataset.attrs = {"PDATA":"radargram data array",
                            "UTMx": "UTM Easting coordinate of each radar trace (x) for EPSG:32611 (UTM Zone 11)", 
                            "UTMy": "UTM Northing coordinate of each radar trace (x) for EPSG:32611 (UTM Zone 11)",
                            "TWT":"two way travel time of the radargram in seconds",
                           } # add attributes

        # write the Dataset to a NetCDF file
        dataset.to_netcdf(datapath+file[:-3]+'_v1.nc', mode='w', engine='netcdf4')

/Users/jukesliu/Documents/POSTDOC/snow-radar/data_submission/sample_data_RME/p1_16.nc
/Users/jukesliu/Documents/POSTDOC/snow-radar/data_submission/sample_data_RME/pt3_31.nc
/Users/jukesliu/Documents/POSTDOC/snow-radar/data_submission/sample_data_RME/pdriftzigzag_14.nc
